# Lab 04.2 — STM vs LTM e Semantic Search

## Overview

We will diferenciar:

- **STM (Short-Term Memory)** — raw events from the current session
- **LTM (Long-Term Memory)** — facts/preferences extracted automatically,
  buscáveis semanticamente

Operações:
- `create_event` → grava em STM
- `list_events` → reads STM from the session
- `retrieve_memories` → busca semântica em LTM

## Prerequisites
- ✅ Lab 04.1 (Memory criado e ACTIVE)

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
from shared.utils.config import load_config, get_region, get_sector
from utils import create_event, list_events, retrieve_memories

cfg = load_config()
region = get_region()
sector = get_sector()
memory_id = cfg["MEMORY_ID"]
print(f"Memory: {memory_id}")

## Step 1: Record 3 events in STM (Ana's session)

In [ ]:
# session_id precisa ter >=33 chars
import uuid
session_id = f"ana-session-{uuid.uuid4()}"[:64]
session_id = (session_id + "0" * 33)[:64]  # padding se curto
actor_id = "ana-operadora"  # actorId: só [a-zA-Z0-9-_/] — use username sem @domínio

events_to_create = [
    [{"conversational": {"role": "USER", "content": {"text": "I want to know the status of the east sector"}}}],
    [{"conversational": {"role": "ASSISTANT", "content": {"text": "East sector on alert — 210.5 MW, voltage 137.8 kV"}}}],
    [{"conversational": {"role": "USER", "content": {"text": "Prefiro respostas em formato bullet"}}}],
]

for payload in events_to_create:
    eid = create_event(memory_id, actor_id=actor_id, session_id=session_id, payload=payload, region=region)
    role = payload[0]["conversational"]["role"]
    print(f"  ✓ Evento {role}: {eid}")

## Step 2: List session events (STM)

In [ ]:
events = list_events(memory_id, actor_id=actor_id, session_id=session_id, region=region)
print(f"\n{len(events)} events in session:\n")
for ev in events:
    p = ev.get("payload", [{}])[0].get("conversational", {})
    role = p.get("role", "?")
    text = p.get("content", {}).get("text", "")[:80]
    print(f"  {role}: {text}")

## Step 3: Aguardar processamento de LTM

LTM is populated **asynchronously** after events are recorded. In production,
isso leva 30-90 segundos. Para o workshop, we will esperar 60s e tentar buscar.

In [ ]:
import time
print("Aguardando 60s para o pipeline LTM processar...")
time.sleep(60)
print("Pronto.")

## Step 4: Semantic search in LTM

In [ ]:
# Search for facts about sectors
namespace_facts = f"/{sector}/facts/{actor_id}"
results = retrieve_memories(
    memory_id=memory_id,
    namespace=namespace_facts,
    query="east sector status",
    top_k=5,
    region=region,
)
print(f"\n{len(results)} memórias encontradas:\n")
for r in results:
    score = r.get("score", "?")
    content = r.get("content", {}).get("text", "")[:100]
    print(f"  [{score}] {content}")

## Step 5: Search user preferences

In [ ]:
namespace_prefs = f"/{sector}/preferences/{actor_id}"
prefs = retrieve_memories(
    memory_id=memory_id,
    namespace=namespace_prefs,
    query="formato de resposta preferido",
    top_k=3,
    region=region,
)
print(f"\n{len(prefs)} preferences:\n")
for p in prefs:
    print(f"  • {p.get('content', {}).get('text', '')[:100]}")

## 🎓 What you learned

- **STM** = raw events from the current session (`create_event` / `list_events`)
- **LTM** = fatos extraídos por estratisgias (`retrieve_memory_records`)
- LTM has latency (~30-90s for the pipeline to process)
- Namespaces com `{actorId}` garanhas isolamento

## Cleanup

```python
from utils import cleanup_memory
cleanup_memory(memory_id, region=region)
```

## Next

➡️ [Lab 05 — AgentCore Runtime](../05-AgentCore-Runtime/) — Onde Memory + Identity + Gateway se juntam!